# RFDB force-label extraction

This notebook builds a training-label table from RFDB's publicly displayed ride and recording statistics. It does **not** use RFDB's sign-in-protected CSV download route.

The output is one row per reliable recording, with vertical, lateral, longitudinal, and combined force extrema. It also creates a coaster-level table by aggregating recordings.

## 1. Settings

Run a small pilot first. Increase `MAX_RIDES` only after you inspect the result. The one-second delay is intentional, so the collection is considerate of RFDB.

In [1]:
from pathlib import Path

OUTPUT_DIR = Path("rfdb_data")
MAX_RIDES = 755              # Set to None for every coaster after the pilot succeeds.
MAX_WORKERS = 3             # Keep this small to avoid a request burst.
REQUEST_DELAY_SECONDS = 1.0 # Minimum spacing between request starts across all workers.
INCLUDE_UNRELIABLE = False  # RFDB quality < 10 is excluded by default.
INCLUDE_NONCOASTERS = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Imports and helper functions

This uses only Python's standard library—no package installation is required.

In [2]:
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import random
import time
from threading import Lock
from datetime import datetime, timezone
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_URL = "https://rideforcesdb.com"
RFDB_EPOCH = 978_307_200  # Date.UTC(2001, 0, 1), as used by RFDB's public UI
STAT_FIELDS = (
    "miny", "maxy", "minx", "maxx", "minz", "maxz", "minc", "maxc",
    "floater", "flojector", "ejector",
)
request_lock = Lock()
next_request_start = 0.0

def wait_for_request_slot():
    global next_request_start
    with request_lock:
        now = time.monotonic()
        wait = max(0, next_request_start - now)
        next_request_start = max(now, next_request_start) + REQUEST_DELAY_SECONDS
    if wait:
        time.sleep(wait)

def get_json(path, attempts=4):
    """Retrieve a public RFDB JSON response with conservative retries."""
    request = Request(
        BASE_URL + path,
        headers={"User-Agent": "Cal-Poly-course-project/RFDB-extractor"},
    )
    for attempt in range(attempts):
        try:
            wait_for_request_slot()
            with urlopen(request, timeout=30) as response:
                if "application/json" not in response.headers.get("Content-Type", ""):
                    raise RuntimeError(f"Expected JSON from {path}; got {response.headers.get('Content-Type')}")
                return json.load(response)
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as error:
            if attempt == attempts - 1:
                raise RuntimeError(f"Could not retrieve {path}: {error}") from error
            time.sleep((2 ** attempt) + random.uniform(0, 0.5))

def rfdb_time(raw_seconds):
    if raw_seconds in (None, ""):
        return ""
    return datetime.fromtimestamp(float(raw_seconds) + RFDB_EPOCH, tz=timezone.utc).isoformat()

## 3. Fetch the public index and recording labels

Each RFDB ride-index entry points to a ride-info response, which contains its individual recordings and the public peak-force statistics. A `quality` value of 10 is RFDB's normal/reliable category.

In [3]:
rides = get_json("/browse/getrides")
if not INCLUDE_NONCOASTERS:
    rides = [ride for ride in rides if str(ride.get("coaster")) == "1"]
rides = sorted(rides, key=lambda ride: int(ride["recid"]))
if MAX_RIDES is not None:
    rides = rides[:MAX_RIDES]

extracted_at = datetime.now(timezone.utc).isoformat()
rows, failures, seen_recordings = [], [], set()

def fetch_ride(ride_index):
    recid = int(ride_index["recid"])
    try:
        return ride_index, get_json(f"/rideInfo?id={recid}"), None
    except RuntimeError as error:
        return ride_index, None, str(error)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_ride, ride) for ride in rides]
    results = [future.result() for future in as_completed(futures)]

for position, (ride_index, detail, error) in enumerate(results, start=1):
    recid = int(ride_index["recid"])
    if error:
        failures.append({"recid": recid, "ride": ride_index.get("name"), "error": error})
        print(f"[{position}/{len(rides)}] FAILED: {ride_index.get('name')}")
        continue

    for recording in detail.get("recordings", []):
        recording_id = int(recording["id"])
        if recording_id in seen_recordings:
            continue
        seen_recordings.add(recording_id)
        quality = recording.get("quality")
        if not INCLUDE_UNRELIABLE and quality != 10:
            continue
        stats = recording.get("statistics") or {}
        row = {
            "recording_id": recording_id,
            "ride_name": detail.get("name", ride_index.get("name", "")),
            "park": detail.get("location", ride_index.get("location", "")),
            "manufacturer": ride_index.get("manufacturer", ""),
            "is_coaster": ride_index.get("coaster", ""),
            "seat": recording.get("seat", ""),
            "advanced_recording": recording.get("advanced", ""),
            "quality": quality,
            "recorded_at_utc": rfdb_time(recording.get("date")),
            "source_url": f"{BASE_URL}/?id={recording_id}",
            "extracted_at_utc": extracted_at,
        }
        row.update({field: stats.get(field, "") for field in STAT_FIELDS})
        rows.append(row)

    print(f"[{position}/{len(rides)}] {detail.get('name', recid)}")
print(f"Collected {len(rows)} reliable recordings from {len(rides)} rides.")
print(f"Failed ride requests: {len(failures)}")

[1/751] Zippin Pippin
[2/751] Roller Coaster
[3/751] Giant Dipper
[4/751] Flight of the Himalayan Eagle
[5/751] Freischütz
[6/751] Abismo
[7/751] Spinball Whizzer
[8/751] Fridolin's verrückter Zauberexpress
[9/751] Wicker Man
[10/751] Family Inverted Coaster
[11/751] Stunt Fall
[12/751] Mandrill Mayhem
[13/751] Wild Mouse
[14/751] Wacky Worm
[15/751] Wildcat
[16/751] Boomerang
[17/751] Galaxy Spin
[18/751] Roller Skater
[19/751] Fury
[20/751] Batman: The Ride
[21/751] Wicked Twister
[22/751] Whistle Punk Chaser
[23/751] Wilderness Run
[24/751] Dragon's Fury
[25/751] Galactica
[26/751] Minifigure Speedway - All Stars
[27/751] Minifigure Speedway - Legends
[28/751] Hoosier Hurricane
[29/751] Steel Hawg
[30/751] Tig'rr
[31/751] Cyclone
[32/751] SpongeBob SquarePants: Rock Bottom Plunge
[33/751] Little Dipper
[34/751] Fairly Odd Coaster
[35/751] Kid Flash
[36/751] Super Grover's Box Car Derby
[37/751] Top Thrill Dragster
[38/751] Vortex
[39/751] Nighthawk
[40/751] Raptor
[41/751] Booster B

## 4. Save the recording-level dataset

The raw public index and failures are retained to make the extraction reproducible.

In [4]:
fields = [
    "recording_id", "ride_name", "park", "manufacturer", "is_coaster", "seat",
    "advanced_recording", "quality", "recorded_at_utc", *STAT_FIELDS,
    "source_url", "extracted_at_utc",
]
recording_path = OUTPUT_DIR / "rfdb_recording_labels.csv"
with recording_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

(OUTPUT_DIR / "rfdb_ride_index_raw.json").write_text(json.dumps(rides, indent=2), encoding="utf-8")
(OUTPUT_DIR / "rfdb_failures.json").write_text(json.dumps(failures, indent=2), encoding="utf-8")
print(f"Saved: {recording_path.resolve()}")

Saved: C:\Users\Jamie\Documents\Codex\2026-11-05\pl\outputs\rfdb_data\rfdb_recording_labels.csv


## 5. Create coaster-level labels for model training

For a model that predicts forces from coaster design specifications, one row per coaster is usually safer than treating repeat recordings as independent observations. This cell calculates the median extrema and an interquartile range (IQR) for each target.

In [5]:
from collections import defaultdict
from statistics import median

def percentile(values, p):
    values = sorted(values)
    if len(values) == 1:
        return values[0]
    position = (len(values) - 1) * p
    lower, upper = int(position), min(int(position) + 1, len(values) - 1)
    return values[lower] + (values[upper] - values[lower]) * (position - lower)

by_ride = defaultdict(list)
for row in rows:
    by_ride[(row["ride_name"], row["park"])].append(row)

ride_rows = []
for (ride_name, park), recordings in by_ride.items():
    ride_row = {
        "ride_name": ride_name,
        "park": park,
        "manufacturer": recordings[0]["manufacturer"],
        "valid_recording_count": len(recordings),
    }
    for field in STAT_FIELDS:
        values = [float(recording[field]) for recording in recordings if recording[field] != ""]
        ride_row[f"{field}_median"] = median(values) if values else ""
        ride_row[f"{field}_iqr"] = percentile(values, 0.75) - percentile(values, 0.25) if len(values) > 1 else 0
    ride_rows.append(ride_row)

ride_fields = list(ride_rows[0]) if ride_rows else ["ride_name", "park", "manufacturer", "valid_recording_count"]
ride_path = OUTPUT_DIR / "rfdb_coaster_labels.csv"
with ride_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=ride_fields)
    writer.writeheader()
    writer.writerows(ride_rows)

print(f"Saved {len(ride_rows)} coaster-level rows: {ride_path.resolve()}")
ride_rows[:3]

Saved 751 coaster-level rows: C:\Users\Jamie\Documents\Codex\2026-11-05\pl\outputs\rfdb_data\rfdb_coaster_labels.csv


[{'ride_name': 'Zippin Pippin',
  'park': 'Bay Beach',
  'manufacturer': 'Martin & Vleminckx',
  'valid_recording_count': 4,
  'miny_median': -0.819,
  'miny_iqr': 0.204175,
  'maxy_median': 2.96365,
  'maxy_iqr': 0.025274999999999714,
  'minx_median': -1.10205,
  'minx_iqr': 0.11135000000000006,
  'maxx_median': 1.10325,
  'maxx_iqr': 0.1921750000000001,
  'minz_median': -0.69355,
  'minz_iqr': 0.137725,
  'maxz_median': 0.56775,
  'maxz_iqr': 0.1028,
  'minc_median': 0.01905,
  'minc_iqr': 0.005200000000000003,
  'maxc_median': 2.98075,
  'maxc_iqr': 0.03167500000000034,
  'floater_median': 4.66,
  'floater_iqr': 0.3200000000000003,
  'flojector_median': 0.18,
  'flojector_iqr': 0.14500000000000002,
  'ejector_median': 0.06,
  'ejector_iqr': 0.075},
 {'ride_name': 'Roller Coaster',
  'park': 'Lagoon',
  'manufacturer': 'John Miller',
  'valid_recording_count': 1,
  'miny_median': -0.3504,
  'miny_iqr': 0,
  'maxy_median': 3.0154,
  'maxy_iqr': 0,
  'minx_median': -0.6079,
  'minx_iqr

## 6. Match the Kaggle roller-coaster database to the RFDB labels

This section retains only Kaggle rows that can be matched conservatively to a coaster in `rfdb_coaster_labels.csv`. It uses normalized **ride name + park** when a park column is available. That is much safer than title-only matching because coaster names repeat.

If the imports fail, install them in the VS Code notebook kernel with `%pip install kagglehub pandas`. KaggleHub may also prompt you to authenticate.

In [14]:
import re
import unicodedata
from pathlib import Path

try:
    import kagglehub
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])
    import kagglehub
import pandas as pd

KAGGLE_DATASET = "robikscube/rollercoaster-database"
KAGGLE_FILE = None  # Set this to a specific CSV path if the automatic choice is not the right file.

kaggle_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
csv_files = sorted(kaggle_path.rglob("*.csv"), key=lambda file: file.stat().st_size, reverse=True)
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {kaggle_path}")

source_file = Path(KAGGLE_FILE) if KAGGLE_FILE else csv_files[0]
kaggle_df = pd.read_csv(source_file)
print(f"Using: {source_file}")
print(f"Kaggle rows: {len(kaggle_df):,}")
print("Columns:", kaggle_df.columns.tolist())
kaggle_df.head()

Using: C:\Users\Jamie\.cache\kagglehub\datasets\robikscube\rollercoaster-database\versions\2\coaster_db.csv
Kaggle rows: 1,087
Columns: ['coaster_name', 'Length', 'Speed', 'Location', 'Status', 'Opening date', 'Type', 'Manufacturer', 'Height restriction', 'Model', 'Height', 'Inversions', 'Lift/launch system', 'Cost', 'Trains', 'Park section', 'Duration', 'Capacity', 'G-force', 'Designer', 'Max vertical angle', 'Drop', 'Soft opening date', 'Fast Lane available', 'Replaced', 'Track layout', 'Fastrack available', 'Soft opening date.1', 'Closing date', 'Opened', 'Replaced by', 'Website', 'Flash Pass Available', 'Must transfer from wheelchair', 'Theme', 'Single rider line available', 'Restraint Style', 'Flash Pass available', 'Acceleration', 'Restraints', 'Name', 'year_introduced', 'latitude', 'longitude', 'Type_Main', 'opening_date_clean', 'speed1', 'speed2', 'speed1_value', 'speed1_unit', 'speed_mph', 'height_value', 'height_unit', 'height_ft', 'Inversions_clean', 'Gforce_clean']


,coaster_name,Length,Speed,Location,Status,Opening date,Type,Manufacturer,Height restriction,Model,...,speed1,speed2,speed1_value,speed1_unit,speed_mph,height_value,height_unit,height_ft,Inversions_clean,Gforce_clean
0,Switchback Railway,600 ft (180 m),6 mph (9.7 km/h),Coney Island,Removed,"June 16, 1884",Wood,LaMarcus Adna Thompson,NaN,Lift Packed,...,6 mph,9.7 km/h,6.0,mph,6.0,50.0,ft,NaN,0,2.9
1,Flip Flap Railway,NaN,NaN,Sea Lion Park,Removed,1895,Wood,Lina Beecher,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,12.0
2,Switchback Railway (Euclid Beach Park),NaN,NaN,"Cleveland, Ohio, United States",Closed,NaN,Other,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
3,Loop the Loop (Coney Island),NaN,NaN,Other,Removed,1901,Steel,Edwin Prescott,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN
4,Loop the Loop (Young's Pier),NaN,NaN,Other,Removed,1901,Steel,Edwin Prescott,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN


### Select the matching columns

The next cell tries common column names automatically. Read its output. If either detected column is wrong, set `RIDE_NAME_COLUMN` and `PARK_COLUMN` explicitly, then rerun this cell and the following cells.

In [15]:
RIDE_NAME_COLUMN = None  # Example: "Name"
PARK_COLUMN = None       # Example: "Park"; leave as None only if no park field exists.

def compact_column_name(value):
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9]+", "", text)

def find_column(columns, preferred):
    lookup = {compact_column_name(column): column for column in columns}
    for candidate in preferred:
        if candidate in lookup:
            return lookup[candidate]
    for compact, original in lookup.items():
        if any(candidate in compact for candidate in preferred):
            return original
    return None

ride_column = RIDE_NAME_COLUMN or find_column(
    kaggle_df.columns, ["coastername", "rollercoastername", "name", "title"]
)
park_column = PARK_COLUMN or find_column(
    kaggle_df.columns, ["park", "amusementpark", "location"]
)
if ride_column is None:
    raise ValueError("Could not identify the Kaggle ride-name column. Set RIDE_NAME_COLUMN above.")

print(f"Ride-name column: {ride_column!r}")
print(f"Park column: {park_column!r}")
if park_column is None:
    print("WARNING: no park column selected; only unambiguous ride names will be accepted.")

Ride-name column: 'coaster_name'
Park column: 'Location'


In [16]:
def normalize_key(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]+", "", str(value).casefold())

rfdb_df = pd.read_csv(ride_path)
rfdb_df["_ride_key"] = rfdb_df["ride_name"].map(normalize_key)
rfdb_df["_park_key"] = rfdb_df["park"].map(normalize_key)
kaggle_match = kaggle_df.copy()
kaggle_match["_ride_key"] = kaggle_match[ride_column].map(normalize_key)

if park_column:
    kaggle_match["_park_key"] = kaggle_match[park_column].map(normalize_key)
    merge_keys = ["_ride_key", "_park_key"]
else:
    # Do not automatically accept duplicate coaster titles without a park.
    merge_keys = ["_ride_key"]
    rfdb_df = rfdb_df[~rfdb_df.duplicated(merge_keys, keep=False)].copy()

rfdb_features = rfdb_df.drop(columns=["ride_name", "park", "manufacturer"], errors="ignore")
rfdb_features = rfdb_features.rename(columns={column: f"rfdb_{column}" for column in rfdb_features.columns if column not in merge_keys})
matched = kaggle_match.merge(rfdb_features, on=merge_keys, how="inner", validate="many_to_one")
unmatched_kaggle = kaggle_match.merge(rfdb_features[merge_keys].drop_duplicates(), on=merge_keys, how="left", indicator=True)
unmatched_kaggle = unmatched_kaggle.loc[unmatched_kaggle["_merge"] == "left_only"].drop(columns="_merge")

matched_path = OUTPUT_DIR / "kaggle_rows_matched_to_rfdb.csv"
review_path = OUTPUT_DIR / "kaggle_rows_unmatched_review.csv"
matched.to_csv(matched_path, index=False)
unmatched_kaggle.to_csv(review_path, index=False)

print(f"Matched Kaggle rows: {len(matched):,}")
print(f"Unmatched Kaggle rows for review: {len(unmatched_kaggle):,}")
print(f"Saved matched dataset: {matched_path.resolve()}")
matched.head()

Matched Kaggle rows: 135
Unmatched Kaggle rows for review: 952
Saved matched dataset: C:\Users\Jamie\Documents\Codex\2026-11-05\pl\outputs\rfdb_data\kaggle_rows_matched_to_rfdb.csv


,coaster_name,Length,Speed,Location,Status,Opening date,Type,Manufacturer,Height restriction,Model,...,rfdb_minc_median,rfdb_minc_iqr,rfdb_maxc_median,rfdb_maxc_iqr,rfdb_floater_median,rfdb_floater_iqr,rfdb_flojector_median,rfdb_flojector_iqr,rfdb_ejector_median,rfdb_ejector_iqr
0,Leap-The-Dips,"1,452 ft (443 m)",10 mph (16 km/h),Lakemont Park,Operating,1902,Wood – Side friction,Federal Construction Company,NaN,NaN,...,0.03860,0.00000,2.71770,0.00000,1.00,0.00,0.00,0.00,0.0,0.0
1,Giant Dipper,"2,640 ft (800 m)",55 mph (89 km/h),Santa Cruz Beach Boardwalk,Operating,"May 17, 1924",Wood,Arthur Looff,50 in (127 cm),NaN,...,0.06335,0.00825,3.71375,0.00755,3.07,0.49,0.21,0.05,0.0,0.0
2,Yankee Cannonball,"609.6 m (2,000 ft)",56.3 km/h (35.0 mph),Canobie Lake Park,Operating,1930,Wood – Out and back,Philadelphia Toboggan Coasters,NaN,NaN,...,0.03660,0.00000,3.31030,0.00000,2.78,0.00,0.00,0.00,0.0,0.0
3,Dahlonega Mine Train,"2,327 ft (709 m)",29 mph (47 km/h),Six Flags Over Georgia,Operating,"June 16, 1967",Steel – Mine Train,Arrow Dynamics,42 in (107 cm),Mine Train,...,0.14730,0.03180,2.42820,0.10760,0.24,0.35,0.00,0.00,0.0,0.0
4,Cedar Creek Mine Ride,"2,540 ft (770 m)",42 mph (68 km/h),Cedar Point,Operating,"May 24, 1969",Steel – Mine Train,Arrow Development,48 in (122 cm),Mine Train,...,0.41530,0.08300,2.26890,0.18040,0.06,0.22,0.00,0.00,0.0,0.0


The matched export contains the original Kaggle specification columns plus `rfdb_...` columns containing the G-force labels. Before accepting fuzzy title matches, manually inspect them—name-only fuzzy matching can accidentally join different coasters with the same or similar titles.

### Expanded title-alias candidates

This covers title matches such as `Ride (roller coaster)` and `Ride (Park)` by stripping parenthetical qualifiers. It accepts only one-to-one title or title-and-park pairs automatically; all other candidate pairs are saved for review.

In [17]:
from collections import defaultdict

def alias_key(value):
    value = re.sub(r"\s*\([^)]*\)", "", str(value))
    return normalize_key(value)

rfdb_alias = rfdb_df.copy().reset_index(names="rfdb_row")
kaggle_alias = kaggle_match.copy().reset_index(names="kaggle_row")
rfdb_alias["title_alias"] = rfdb_alias["ride_name"].map(alias_key)
kaggle_alias["title_alias"] = kaggle_alias[ride_column].map(alias_key)
if park_column:
    kaggle_alias["park_alias"] = kaggle_alias[park_column].map(normalize_key)

rfdb_groups, kaggle_groups = defaultdict(list), defaultdict(list)
for index, row in rfdb_alias.iterrows():
    rfdb_groups[row.title_alias].append(index)
for index, row in kaggle_alias.iterrows():
    kaggle_groups[row.title_alias].append(index)

accepted, candidates = [], []
for title in sorted(set(rfdb_groups) & set(kaggle_groups)):
    rfdb_ids, kaggle_ids = rfdb_groups[title], kaggle_groups[title]
    rfdb_park_counts = defaultdict(int)
    kaggle_park_counts = defaultdict(int)
    for index in rfdb_ids: rfdb_park_counts[rfdb_alias.at[index, "_park_key"]] += 1
    if park_column:
        for index in kaggle_ids: kaggle_park_counts[kaggle_alias.at[index, "park_alias"]] += 1
    for rfdb_id in rfdb_ids:
        for kaggle_id in kaggle_ids:
            same_park = bool(park_column) and rfdb_alias.at[rfdb_id, "_park_key"] == kaggle_alias.at[kaggle_id, "park_alias"]
            auto = (len(rfdb_ids) == len(kaggle_ids) == 1) or (same_park and rfdb_park_counts[rfdb_alias.at[rfdb_id, "_park_key"]] == kaggle_park_counts[kaggle_alias.at[kaggle_id, "park_alias"]] == 1)
            candidate = {
                "title_alias": title, "kaggle_row": kaggle_alias.at[kaggle_id, "kaggle_row"],
                "kaggle_title": kaggle_alias.at[kaggle_id, ride_column],
                "kaggle_park": kaggle_alias.at[kaggle_id, park_column] if park_column else "",
                "rfdb_row": rfdb_alias.at[rfdb_id, "rfdb_row"],
                "rfdb_title": rfdb_alias.at[rfdb_id, "ride_name"],
                "rfdb_park": rfdb_alias.at[rfdb_id, "park"],
                "rfdb_manufacturer": rfdb_alias.at[rfdb_id, "manufacturer"],
                "same_park": same_park, "auto_accept": auto,
            }
            (accepted if auto else candidates).append(candidate)

accepted_path = OUTPUT_DIR / "rfdb_kaggle_alias_auto_matches.csv"
review_path = OUTPUT_DIR / "rfdb_kaggle_alias_review_candidates.csv"
pd.DataFrame(accepted).to_csv(accepted_path, index=False)
pd.DataFrame(candidates).to_csv(review_path, index=False)
print(f"Shared normalized title aliases: {len(set(rfdb_groups) & set(kaggle_groups)):,}")
print(f"Automatically accepted pairs: {len(accepted):,}")
print(f"Ambiguous candidate pairs to review: {len(candidates):,}")
print(f"Saved automatic matches: {accepted_path.resolve()}")
pd.DataFrame(accepted).head()

Shared normalized title aliases: 400
Automatically accepted pairs: 373
Ambiguous candidate pairs to review: 609
Saved automatic matches: C:\Users\Jamie\Documents\Codex\2026-11-05\pl\outputs\rfdb_data\rfdb_kaggle_alias_auto_matches.csv


,title_alias,kaggle_row,kaggle_title,kaggle_park,rfdb_row,rfdb_title,rfdb_park,rfdb_manufacturer,same_park,auto_accept
0,abyssus,1057,Abyssus,Energylandia,414,Abyssus,Energylandia,Vekoma,True,True
1,adrenalinepeak,1000,Adrenaline Peak,Oaks Amusement Park,41,Adrenaline Peak,Oaks Park,Gerstlauer Amusement Rides GmbH,False,True
2,adventureexpress,299,Adventure Express,Kings Island,518,Adventure Express,Kings Island,Arrow Dynamics,True,True
3,airgrover,805,Air Grover,"Tampa, Florida, U.S.",290,Air Grover,Busch Gardens Tampa,Zierer,False,True
4,alpenexpressenzian,205,Alpenexpress Enzian,Europa-Park,698,Alpenexpress Enzian,Europa Park,Mack Rides GmbH & Co KG,True,True


## Notes

- `y`, `x`, `z`, and `c` correspond to RFDB's vertical, lateral, longitudinal, and combined-force values.
- RFDB warns that calibration, device movement, and smoothing affect accuracy. Treat the labels as observational estimates—not certified design or safety measurements.
- Keep `recording_id` and `source_url` in the recording-level file so individual observations remain auditable.
- When you join to coaster specifications later, split train/test data by coaster, not by recording.